# Stress Prediction — Final Submission Notebook

본 노트북은 **Data Leakage 방지 규정과 코드 재현성**을 고려하여 최종 제출용으로 정리한 버전입니다.

- 모든 통계 기반 전처리(IQR 경계, 결측치 중앙값, StandardScaler, OneHotEncoder)는 **Train 데이터에만 fit**합니다.
- Validation/Test에는 Train에서 계산한 값과 fit된 변환기만 적용합니다.
- 파생 변수는 각 행(row) 내부의 값만 사용하여 생성합니다.
- 하이퍼파라미터 탐색과 모델 선택은 **Train 데이터의 5-Fold CV 결과만** 사용합니다.
- Test 데이터는 최종 추론 및 제출 파일 생성 단계에서만 로드합니다.
- 외부 데이터 및 사전 학습 모델은 사용하지 않습니다.

최종 설정:
`tree_q=0.54`, `pair_q=0.42`, `pair_weight=0.24`, `override_threshold=0.15`


In [ ]:
from __future__ import annotations

import os
import platform
import sys
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
import sklearn
import scipy
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from scipy.spatial import cKDTree

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200

# 재현 가능한 최종 하이퍼파라미터: Train 5-Fold CV만으로 선택
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.42
PAIR_WEIGHT = 0.24
OVERRIDE_THRESHOLD = 0.15
WINSORIZE_COLUMNS = ["mean_working", "diastolic_blood_pressure", "cholesterol", "glucose"]

# 대회 제출 규칙에 맞춰 모든 데이터 입/출력 경로를 /data로 통일
DATA_DIR = Path("/data")
OUTPUT_DIR = Path("/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def compute_iqr_bounds(train_frame, columns):
    """Train 데이터만 사용해 IQR 기반 clipping 경계를 계산합니다."""
    bounds = {}
    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame, bounds):
    """Train에서 계산된 경계를 현재 데이터에 적용합니다."""
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """각 행 내부 값만 사용하여 파생 변수를 생성합니다."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    """Train fold의 컬럼 타입을 기준으로 전처리 파이프라인을 구성합니다."""
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_raw_predictions(train_features, val_features, target) -> np.ndarray:
    """한 번만 fit하고, 개별 트리 예측 행렬(n_val, n_estimators)을 그대로 반환합니다."""
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    return np.column_stack([t.predict(val_matrix) for t in model.estimators_])


def empirical_rank_transform(train_values, inference_values):
    """Train 분포만으로 경험적 순위를 계산하고 추론 데이터에 동일 기준을 적용합니다."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    inference_values = np.nan_to_num(inference_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    inference_rank = np.searchsorted(sorted_train, inference_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), inference_rank.astype(np.float32)


def pair_neighbor_targets(train_features, val_features, target) -> np.ndarray:
    """모든 컬럼쌍(28개)에 대한 최근접 이웃 타겟 행렬(n_val, n_pairs)을 반환합니다."""
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.column_stack(neighbor_targets)


def nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols):
    train_numeric = train_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    val_numeric = val_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)
    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


def combined_prediction(train_raw_fold, val_raw_fold, target, numeric_cols,
                         tree_q=TREE_QUANTILE, pair_q=PAIR_QUANTILE,
                         pair_weight=PAIR_WEIGHT, override_threshold=OVERRIDE_THRESHOLD):
    """Train 기준 전처리 후 ExtraTrees·pair-NN·근접이웃 보정을 결합해 예측합니다."""
    bounds = compute_iqr_bounds(train_raw_fold, WINSORIZE_COLUMNS)
    train_winsorized = apply_winsorize(train_raw_fold, bounds)
    val_winsorized = apply_winsorize(val_raw_fold, bounds)

    train_feat = add_features(train_winsorized)
    val_feat = add_features(val_winsorized)

    tree_raw = tree_raw_predictions(train_feat, val_feat, target)
    pair_raw = pair_neighbor_targets(train_feat, val_feat, target)
    tree_pred = np.quantile(tree_raw, tree_q, axis=1)
    pair_pred = np.quantile(pair_raw, pair_q, axis=1)
    base_pred = np.clip(np.round((1 - pair_weight) * tree_pred + pair_weight * pair_pred, 2), 0.0, 1.0)

    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, val_raw_fold, target, numeric_cols)
    override_mask = nearest_dist < override_threshold
    result = base_pred.copy()
    result[override_mask] = np.clip(np.round(nearest_target[override_mask], 2), 0.0, 1.0)
    return result


print("=== 실행 환경 ===")
print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)
print()

print("설정 로드 완료:", dict(tree_q=TREE_QUANTILE, pair_q=PAIR_QUANTILE,
                          pair_weight=PAIR_WEIGHT, override_threshold=OVERRIDE_THRESHOLD))


## Train-only 하이퍼파라미터 탐색

5-Fold CV의 각 fold에서 학습 fold만 사용해 전처리와 모델을 fit하고 validation fold를 예측합니다.
폴드별 원시 예측을 캐시한 뒤 quantile, 가중치, override threshold를 비교하므로 **Test 데이터는 이 단계에 사용되지 않습니다.**

탐색 순서: ① `PAIR_WEIGHT` → ② `TREE_QUANTILE × PAIR_QUANTILE` → ③ `PAIR_WEIGHT` 재탐색 → ④ `OVERRIDE_THRESHOLD`.
각 후보는 동일한 Train-only OOF MAE로 비교합니다.


In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
y = train_raw[TARGET].reset_index(drop=True)
raw_input = train_raw.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
numeric_cols = raw_input.select_dtypes(include=["number"]).columns.tolist()

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)

fold_cache = []
for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_raw_fold = raw_input.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_input.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    bounds = compute_iqr_bounds(train_raw_fold, WINSORIZE_COLUMNS)
    train_winsorized = apply_winsorize(train_raw_fold, bounds)
    val_winsorized = apply_winsorize(val_raw_fold, bounds)
    train_feat = add_features(train_winsorized)
    val_feat = add_features(val_winsorized)

    tree_raw = tree_raw_predictions(train_feat, val_feat, y_train)
    pair_raw = pair_neighbor_targets(train_feat, val_feat, y_train)
    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, val_raw_fold, y_train, numeric_cols)

    fold_cache.append((val_idx, tree_raw, pair_raw, nearest_dist, nearest_target))
    print(f"Fold {fold} 캐시 완료")


def evaluate(tree_q, pair_q, pair_weight, override_threshold):
    oof = np.zeros(len(y))
    for val_idx, tree_raw, pair_raw, nearest_dist, nearest_target in fold_cache:
        tree_pred = np.quantile(tree_raw, tree_q, axis=1)
        pair_pred = np.quantile(pair_raw, pair_q, axis=1)
        pred = np.clip(np.round((1 - pair_weight) * tree_pred + pair_weight * pair_pred, 2), 0.0, 1.0)
        override_mask = nearest_dist < override_threshold
        pred[override_mask] = np.clip(np.round(nearest_target[override_mask], 2), 0.0, 1.0)
        oof[val_idx] = pred
    return mean_absolute_error(y, oof)


baseline = evaluate(0.54, 0.48, 0.28, 0.2)
print(f"\n[기준 설정] tree_q=0.54 pair_q=0.48 pair_w=0.28 thr=0.2 -> {baseline:.6f}")

print("\n=== ① PAIR_WEIGHT 확장 탐색 (tree_q=0.54, pair_q=0.48, thr=0.2) ===")
stage1 = []
for w in [0.0, 0.05, 0.10, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24]:
    mae = evaluate(0.54, 0.48, w, 0.2)
    stage1.append((w, mae))
    print(f"PAIR_WEIGHT={w:.2f}: {mae:.6f}")
stage1.sort(key=lambda r: r[1])
best_w = stage1[0][0]
print(f"-> 최적: PAIR_WEIGHT={best_w} ({stage1[0][1]:.6f})")

print(f"\n=== ② TREE_QUANTILE x PAIR_QUANTILE 그리드 (pair_w={best_w}, thr=0.2) ===")
stage2 = []
for tq in [0.48, 0.50, 0.52, 0.54, 0.56, 0.58]:
    for pq in [0.42, 0.44, 0.46, 0.48, 0.50, 0.52]:
        mae = evaluate(tq, pq, best_w, 0.2)
        stage2.append((tq, pq, mae))
stage2.sort(key=lambda r: r[2])
for tq, pq, mae in stage2[:5]:
    print(f"tree_q={tq:.2f} pair_q={pq:.2f}: {mae:.6f}")
best_tq, best_pq, _ = stage2[0]
print(f"-> 최적: tree_q={best_tq} pair_q={best_pq}")

print(f"\n=== ③ PAIR_WEIGHT 재탐색 (tree_q={best_tq}, pair_q={best_pq}, thr=0.2) ===")
stage3 = []
for w in [0.0, 0.05, 0.10, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24, 0.28, 0.32]:
    mae = evaluate(best_tq, best_pq, w, 0.2)
    stage3.append((w, mae))
    print(f"PAIR_WEIGHT={w:.2f}: {mae:.6f}")
stage3.sort(key=lambda r: r[1])
best_w2 = stage3[0][0]
print(f"-> 최적: PAIR_WEIGHT={best_w2} ({stage3[0][1]:.6f})")

print(f"\n=== ④ OVERRIDE_THRESHOLD 탐색 (tree_q={best_tq}, pair_q={best_pq}, pair_w={best_w2}) ===")
stage4 = []
for thr in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    mae = evaluate(best_tq, best_pq, best_w2, thr)
    stage4.append((thr, mae))
    print(f"OVERRIDE_THRESHOLD={thr}: {mae:.6f}")
stage4.sort(key=lambda r: r[1])
best_thr, best_final_mae = stage4[0]
print(f"-> 최적: OVERRIDE_THRESHOLD={best_thr} ({best_final_mae:.6f})")

print("\n================ 최종 채택 설정 ================")
print(f"tree_q={best_tq}, pair_q={best_pq}, pair_weight={best_w2}, override_threshold={best_thr}")
print(f"CV MAE: {best_final_mae:.6f}")


## 5-Fold CV — 최종 설정 재검증

선택된 최종 설정을 사용해 전처리부터 모델 학습까지 다시 수행하고 OOF MAE를 확인합니다.
각 validation fold는 해당 fold의 학습 데이터에서 계산한 통계와 변환기만 사용합니다.


In [ ]:
oof_final = np.zeros(len(y))
for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_raw_fold = raw_input.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_input.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    oof_final[val_idx] = combined_prediction(train_raw_fold, val_raw_fold, y_train, numeric_cols)
    print(f"Fold {fold} done")

final_cv_mae = mean_absolute_error(y, oof_final)
print()
print(f"### 최종 설정 CV MAE: {final_cv_mae:.6f}")


## 최종 Test 추론 및 제출 파일 생성

Test 데이터는 이 단계에서 처음 로드합니다. 모든 통계 기반 전처리와 변환기는 전체 Train 데이터에 fit한 뒤 Test에 transform만 적용합니다.
제출 파일의 ID는 제공된 Test ID를 그대로 사용하며, ID 자체는 모델 입력이나 파생 변수에 사용하지 않습니다.


In [ ]:
# 규정 준수를 위해 Test 데이터는 최종 추론 단계에서만 로드합니다.
test_raw = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

test_ids = test_raw[ID_COLUMN].copy()
test_input = test_raw.drop(columns=[ID_COLUMN]).reset_index(drop=True)

# 전체 Train 데이터에서 학습한 전처리/모델로 Test를 한 번만 추론합니다.
final_prediction = combined_prediction(raw_input, test_input, y, numeric_cols)

# 제공된 제출 양식을 유지하고 예측값만 기록합니다.
output = submission.copy()
output[ID_COLUMN] = test_ids
output[TARGET] = final_prediction

submit_path = OUTPUT_DIR / "submit_stress_prediction_final.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("사용된 설정:", dict(
    tree_q=TREE_QUANTILE,
    pair_q=PAIR_QUANTILE,
    pair_weight=PAIR_WEIGHT,
    override_threshold=OVERRIDE_THRESHOLD,
))
print("Prediction range:", final_prediction.min(), final_prediction.max())
print(output.head(10))

# 제출 파일 자체의 기본 무결성만 확인합니다. 이 결과는 학습/전처리/하이퍼파라미터 선택에 사용하지 않습니다.
checks = {
    "ids_unique": output[ID_COLUMN].is_unique,
    "no_missing_values": not output.isna().any().any(),
    "target_in_range_0_1": output[TARGET].between(0, 1).all(),
}
print("\n=== SUBMISSION VALIDATION ===")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
